## Exercises Day 3


### Progenetix Exercise 4. Download and interpret CNV frequency data

Now download the CNV frequency table for:

Example:
- `Small cell carcinoma, NOS`
- filter code: `pgx:icdom-80413`

Tasks:
- download the frequency table
- inspect its shape
- inspect its columns
- Explain why the frequency table has a different shape from the sample table.
    - what one row represents in the sample table
    - what one row represents in the frequency table
    - why these two tables cannot be merged row by row


-------------

One row in the sample table contains: one biological sample (one tumor / patient) containing: sample ID, cancer type, sequencing platform, clinical / metadata information

One row in the CNV frequency table contains: a reference name, one genomic region (or bin) with aggregated statistics containing: chromosome region (e.g. chr1: start–end bin), frequency of CNV events in that region separated into gain/loss frequencies, number (no), and source query

They describe different biological levels:
- Sample table: Unit = individual patient; Data type = metadata; Shape depends on number of samples
- CNV frequency table: Unit = genomic region; Data type = aggregated signal across cohort; 
- Shape depends on genome binning resolution, number of chromosomes × bins per chromosome


The sample table describes individual patients. The CNV frequency table describes genomic regions summarized across patients. They operate on different units and cannot be directly joined row-by-row.

In [9]:
import requests
import pandas as pd
from io import StringIO

freq_urls = {
    "small_cell_carcinoma_icdom": "https://progenetix.org/services/intervalFrequencies/?output=pgxseg&filters=pgx:icdom-80413"
}

freq_tables = {}

for source_query, url in freq_urls.items():
    print(f"Downloading frequency table for {source_query}")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    df_freq = pd.read_csv(StringIO(response.text), sep="\t", comment="#")
    df_freq["source_query"] = source_query
    freq_tables[source_query] = df_freq

# Show the shape of each downloaded frequency table.

for source_query, df_freq in freq_tables.items():
    print(source_query, df_freq.shape)


freq_tables["small_cell_carcinoma_icdom"].head()

freq_tables["small_cell_carcinoma_icdom"].columns

small_cell_carcinoma_icdom (3106, 7)


Index(['reference_name', 'start', 'end', 'gain_frequency', 'loss_frequency',
       'no', 'source_query'],
      dtype='object')


### Classification Exercise 3. Inspect one informative bin
Choose one bin from the interpretation table.

Tasks:
- write down whether that bin favors glioblastoma or ovarian HGSC
- inspect the histogram for that bin
- open the Progenetix collation plot link and describe whether the broad group-level pattern seems consistent with the model result


--------
Bin chosen: chr4_20000000_30000000, chr4:20-30 Mb, 0.314
- Higher CNV values (duplication) favor glioblastoma, since coefficient is > 0
--------
Histogram:
- Duplication (+1): more common glioblastoma 
- No event (0): equally common in both 
- Deletion (-1): More common in ovarian high-grade serous adenocarcinoma
--------
Progenetix collation plot:

![histogram](d3_sc_histo.png)


We can see that Ovarian High-Grade Serous Adenocarcinoma (HGSC) in that region has:

- far more deletions than Glioblastoma (~34 vs ~2 samples)
- fewer duplications than Glioblastoma (~8 vs ~27 samples)
- a similarly high number of no-event samples


In general we can see that this bin shows clear differences between the 2 cancers — particularly in the deletion category, which is almost exclusive to Ovarian HGSC — making it useful as a classifier

Ovarian High-Grade Serous Adenocarcinoma in general shows a notably higher rate of deletions on chr4 compared to Glioblastoma, and fewer duplications
